In [ ]:
#!pip install xgboost joblib scikit-learn pandas numpy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

In [ ]:
np.random.seed(42)
n_muestras = 1000

edad = np.random.randint(40, 85, n_muestras)
sistolica = np.random.randint(100, 180, n_muestras)
diastolica = np.random.randint(60, 110, n_muestras)
toma_medicamento = np.random.choice([0, 1], size=n_muestras, p=[0.3, 0.7]) # 70% s? se lo toma

In [ ]:
# Crear la regla m?dica para la columna objetivo (Crisis: 1 = S?, 0 = No)
# Riesgo si la presi?n es alta y NO tom? medicamento, o si la presi?n es cr?ticamente alta
riesgo_score = (sistolica * 0.5) + (diastolica * 0.3) - (toma_medicamento * 30)
crisis = np.where(riesgo_score > 65, 1, 0)

In [ ]:
# Crear el DataFrame (la tabla de datos)
df = pd.DataFrame({
    'Edad': edad,
    'Sistolica': sistolica,
    'Diastolica': diastolica,
    'Toma_Medicamento': toma_medicamento,
    'Crisis': crisis
})

In [ ]:
# 2. DIVIDIR DATOS (Caracter?sticas X, Objetivo y)
X = df.drop('Crisis', axis=1)
y = df['Crisis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 3. DEFINICI?N DE LOS 3 ALGORITMOS 
modelos = {
    'Regresi?n Log?stica': LogisticRegression(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

In [ ]:
# 4. ENTRENAR Y EVALUAR
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train) # Entrenamiento
    predicciones = modelo.predict(X_test) # Predicci?n
    
    # Calcular m?tricas
    resultados.append({
        'Algoritmo': nombre,
        'Exactitud (Accuracy)': accuracy_score(y_test, predicciones),
        'Precisi?n (Precision)': precision_score(y_test, predicciones),
        'Recall': recall_score(y_test, predicciones),
        'F1-Score': f1_score(y_test, predicciones)
    })

In [ ]:
# Mostrar la tabla comparativa de m?tricas
df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

In [ ]:
# 5. GUARDAR EL GANADOR (XGBoost) para la API
joblib.dump(modelos['XGBoost'], 'mejor_modelo_htas.pkl')
print("\n?Modelo XGBoost guardado exitosamente como 'mejor_modelo_htas.pkl'!")